In [1]:
library(parallel)
library(GenomicFeatures)
set.seed(1234)
library(repr)
library(motifmatchr)
library(fastmatch)
library(qlcMatrix)

library(ensembldb)
library(EnsDb.Hsapiens.v86)
library(AnnotationFilter)

library(Signac)
library(Seurat)
library(JASPAR2024)
library(TFBSTools)
library(BSgenome.Hsapiens.UCSC.hg38)
library(patchwork)
library(ggplot2)
library(Matrix)
library(zoo)
library(tidyr)

source("../multiome_methods/function_calls.r")
source("../multiome_methods/peak_gene_relations.r")
source("../multiome_methods/binding_site_identification.r")
source("../multiome_methods/TF_activity.r")
source("../multiome_methods/signac_utils.r")

library(dplyr)

library(ggplot2)
library(hexbin)
library(RColorBrewer)
library(cowplot)
library(gridExtra)
library(ggExtra)

library(pheatmap)
library(purrr)
library(data.table)
library(Rsamtools)
library(stringi)


Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The follo

In [3]:
object_w_net <- readRDS("/nfs/home/students/m.back/swarm/backend/calc_multiome_scores/multiome_tf_regulation/notebooks_pipeline/objects_AVN_fibroblast/final_object_w_net.rds")

In [4]:
str(object_w_net@misc, max.level = 1)

List of 12
 $ markers.annotated        :'data.frame':	4075 obs. of  8 variables:
 $ peak_stats               :'data.frame':	130 obs. of  51 variables:
 $ peak_stats.filtered      :'data.frame':	122 obs. of  51 variables:
 $ motifs.inProximalPeaks   :List of 23
 $ motifs.inDistalPeaks     :List of 23
 $ motif_enrichment         :'data.frame':	16560 obs. of  16 variables:
 $ filtered.motif_enrichment:'data.frame':	3290 obs. of  16 variables:
 $ motif_stats              :'data.frame':	3290 obs. of  26 variables:
 $ motif2TF                 :List of 720
 $ TF2motif                 :List of 668
 $ seedNetwork              :List of 5
 $ context_subNetwork       :List of 5


In [5]:
colnames(object_w_net@misc$peak_stats.filtered)

[1] "cluster"                                         
 [2] "peak"                                            
 [3] "gene"                                            
 [4] "annotation"                                      
 [5] "regulatorType"                                   
 [6] "signac.scores"                                   
 [7] "signac.zscores"                                  
 [8] "signac.pvalues"                                  
 [9] "t-stat_highly.acc"                               
[10] "p.value-t.test_highly.acc"                       
[11] "p.value-t.test_highly.acc_Bonf"                  
[12] "p.value-t.test_highly.acc_BH"                    
[13] "expr_not_0"                                      
[14] "acc_not_0"                                       
[15] "expr_not_0.given_acc"                            
[16] "expr_and_acc_not_0"                              
[17] "FC.expr_given_acc"                               
[18] "expr_not_0.bg"                                   
[19] "acc_not_0.bg"                                    
[20] "expr_not_0.given_acc.bg"                         
[21] "expr_and_acc_not_0.bg"                           
[22] "FC.expr_given_acc.bg"                            
[23] "expr_not_0.bg_other_peaks.same_cluster"          
[24] "acc_not_0.bg_other_peaks.same_cluster"           
[25] "expr_not_0.given_acc.bg_other_peaks.same_cluster"
[26] "expr_and_acc_not_0.bg_other_peaks.same_cluster"  
[27] "FC.expr_given_acc.bg_other_peaks.same_cluster"   
[28] "n.bg_other_peaks.same_cluster"                   
[29] "expr_not_0.all"                                  
[30] "acc_not_0.all"                                   
[31] "expr_not_0.given_acc.all"                        
[32] "expr_and_acc_not_0.all"                          
[33] "FC.expr_given_acc.all"                           
[34] "expr_not_0.bg_other_peaks.all"                   
[35] "acc_not_0.bg_other_peaks.all"                    
[36] "expr_not_0.given_acc.bg_other_peaks.all"         
[37] "expr_and_acc_not_0.bg_other_peaks.all"           
[38] "FC.expr_given_acc.bg_other_peaks.all"            
[39] "n.bg_other_peaks.all"                            
[40] "high_cor_distal"                                 
[41] "high_cor_distal_zScore"                          
[42] "promotersLinkedToSeed"                           
[43] "distalPeaksLinkedToSeed"                         
[44] "acc_cells_cluster"                               
[45] "delta_expr_given_acc.same_peak_bg"               
[46] "delta_expr_given_acc.other_peaks.same_cluster"   
[47] "delta_expr_given_acc.other_peaks.all"            
[48] "pass_cluster_specific"                           
[49] "pass_global"                                     
[50] "pass_any"                                        
[51] "pass_type"

## explainations peak-stats 
#### Identity columns 
- 'cluster'
- 'peak'
- 'gene'
- 'annotation': seed gene, or module gene (from prior net), or marker gene
- 'regulatorType': proximal: 100-2000 upstream TSS, else distal

#### LinkPeaks stats: across all clusters
- 'signac.scores': The raw Signac link score from Links(object)$score
- 'signac.zscores': The Signac link z-score from Links(object)$zscore
- 'signac.pvalues': The Signac link p-value from Links(object)$pvalue

#### Cluster-specific peak accessibility test columns: conduct_stat_test() using test='t-test', test_activation=TRUE (is this peak more accessible in the target cluster than outside it?)
foreground: cells of cluster,  background: all other cells, alternative: greater
- t-stat_highly.acc: The one-sided t-test statistic for target-cluster accessibility > background accessibility
- p.value-t.test_highly.acc: The raw p-value from that one-sided t-test
- p.value-t.test_highly.acc_Bonf: Bonferroni-adjusted p-value across all unique (cluster, peak) tests
- p.value-t.test_highly.acc_BH: Benjamini–Hochberg adjusted p-value across all unique (cluster, peak) tests

#### Probability / zero-expression columns: zero_expression_stats() and calc_prob_stats(gene_expr, peak_acc)
(n_cells = number of cells in cluster
n_expr = # cells with gene_expr != 0
n_acc = # cells with peak_acc != 0
n_joint = # cells with gene_expr != 0 AND peak_acc != 0)

- expr_not_0 = n_expr / n_cells: Fraction of cells in the target cluster where the gene is expressed
- acc_not_0 = n_acc / n_cells: Fraction of cells in the target cluster where this peak is accessible
- expr_not_0.given_acc = n_joint / n_acc: Conditional probability that the gene is expressed among cells where this peak is accessible, within the target cluster: P(expr != 0 | acc != 0) in the cluster
- expr_and_acc_not_0 = n_joint / n_cells: Joint fraction of target-cluster cells with both gene expression and peak accessibility nonzero: P(expr != 0 AND acc != 0) in the cluster.
- FC.expr_given_acc = expr_not_0.given_acc / expr_not_0: Fold-enrichment of gene expression among accessible cells relative to the cluster baseline expression rate: So values above 1 mean expression is enriched among cells where the peak is open

#### .bg: Background: same peak, other clusters: 
These are computed on the same peak and same gene, but using all cells outside the target cluster.
.all: same peak, All cells:
These are the same metrics as above but computed using all cells for the same gene and same peak.

#### Per-seed peak counts
- 'promotersLinkedToSeed': Number of linked peaks for this (gene, cluster) whose regulatorType == "proximal"
- 'distalPeaksLinkedToSeed': Number of linked peaks for this (gene, cluster) whose regulatorType == "distal"

#### Convenience / derived filter columns
- 'acc_cells_cluster': Estimated number of accessible cells in the target cluster for this peak: acc_cells_cluster = acc_not_0 * cluster_size where cluster_size = table(Idents(object))[cluster]
- 'delta_expr_given_acc.same_peak_bg': Difference in conditional expression between target cluster and background, for the same peak: expr_not_0.given_acc - expr_not_0.given_acc.bg
(- 'delta_expr_given_acc.other_peaks.same_cluster': Difference between the target peak and the “other linked peaks of same gene” summary, inside the target cluster:
expr_not_0.given_acc - expr_not_0.given_acc.bg_other_peaks.same_cluster
'delta_expr_given_acc.other_peaks.all': Difference between the target peak’s across-all-cells conditional expression and the “other linked peaks of same gene” across-all-cells summary: expr_not_0.given_acc.all - expr_not_0.given_acc.bg_other_peaks.all)

#### Final filter flags
- 'pass_cluster_specific'
    - promoter requirement, if enabled: promotersLinkedToSeed > 0
    - p.value-t.test_highly.acc_BH < th
    - t-stat_highly.acc > cluster_t_min
    - acc_cells_cluster >= min.cells
    - expr_and_acc_not_0 >= cluster_expr_given_acc_min
    - expr_not_0.given_acc > expr_given_acc_th
    - FC.expr_given_acc > cluster_fc_min
    - delta_expr_given_acc.same_peak_bg >= cluster_delta_same_peak_bg_min
- 'pass_global'
    - promoter requirement, if enabled
    - signac.zscores >= global_signac_z_min
    - signac.pvalues < global_signac_p_cutoff
    - expr_and_acc_not_0.all >= global_expr_given_acc_min
    - expr_not_0.given_acc.all > expr_given_acc_th
    - FC.expr_given_acc.all > global_fc_min
- 'pass_any': TRUE if pass_cluster_specific | pass_global
- 'pass_type'

In [6]:
summary_col_labels_short <- c(
  gene = "Gene",
  cluster = "Cluster",
  annotation = "Annotation",
  peak = "Peak",
  regulatorType = "Class",
  signac.scores = "Link Score",
  signac.zscores = "Link Z",
  signac.pvalues = "Link P",
  `t-stat_highly.acc` = "Acc. T-stat",
  `p.value-t.test_highly.acc_BH` = "Acc. FDR",
  acc_cells_cluster = "Accessible Cells",
  `expr_not_0.given_acc` = "P(expr|acc), cluster",
  `expr_not_0.given_acc.bg` = "P(expr|acc), bg",
  expr_and_acc_not_0 = "P(expr & acc), cluster",
  expr_and_acc_not_0.all = "P(expr & acc), all",
  FC.expr_given_acc = "Enrichment, cluster",
  FC.expr_given_acc.all = "Enrichment, all",
  delta_expr_given_acc.same_peak_bg = "Delta P(expr|acc)",
  promotersLinkedToSeed = "Promoter Peaks",
  distalPeaksLinkedToSeed = "Distal Peaks",
  pass_type = "Pass Type"
)

In [7]:
# keep only the selected columns, in this order
summary_cols <- c(
  "gene",
  "cluster",
  "annotation",
  "peak",
  "regulatorType",
  "signac.scores",
  "signac.zscores",
  "signac.pvalues",
  "t-stat_highly.acc",
  "p.value-t.test_highly.acc_BH",
  "acc_cells_cluster",
  "expr_not_0.given_acc",
  "expr_not_0.given_acc.bg",
  "expr_and_acc_not_0",
  "expr_and_acc_not_0.all",
  "FC.expr_given_acc",
  "FC.expr_given_acc.all",
  "delta_expr_given_acc.same_peak_bg",
  "promotersLinkedToSeed",
  "distalPeaksLinkedToSeed",
  "pass_type"
)


# filter + rename
peak_stats_summary <- object_w_net@misc$peak_stats.filtered[, summary_cols]
colnames(peak_stats_summary) <- summary_col_labels_short[summary_cols]

peak_stats_summary

,Gene,Cluster,Annotation,Peak,Class,Link Score,Link Z,Link P,Acc. T-stat,Acc. FDR,⋯,"P(expr|acc), cluster","P(expr|acc), bg","P(expr & acc), cluster","P(expr & acc), all","Enrichment, cluster","Enrichment, all",Delta P(expr|acc),Promoter Peaks,Distal Peaks,Pass Type
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr[1d]>
1,DDR2,Fibroblast,colagen_fibral_orga,chr1-162554311-162554647,distal,0.06974769,2.410024,7.975731e-03,6.138812,9.537753e-10,⋯,0.5600000,0.39130435,0.014791337,0.004426896,1.318507,1.947700,0.1686957,0,1,both
2,COLGALT2,Fibroblast,colagen_fibral_orga,chr1-183926959-183927219,distal,0.06488519,3.308303,4.693163e-04,3.078629,1.122332e-03,⋯,0.5333333,0.20000000,0.004226096,0.001196458,2.011155,3.661774,0.3333333,0,3,global
3,COLGALT2,Fibroblast,colagen_fibral_orga,chr1-183958935-183959242,distal,0.05786672,1.943588,2.597258e-02,3.421591,3.434847e-04,⋯,0.4761905,0.12500000,0.005282620,0.001435750,1.795674,2.969006,0.3511905,0,3,cluster_specific
4,COLGALT2,Fibroblast,colagen_fibral_orga,chr1-184021769-184021996,distal,0.08162847,3.742938,9.094059e-05,2.504964,6.454046e-03,⋯,0.6296296,0.03921569,0.008980454,0.002273271,2.374281,2.229927,0.5904139,0,3,both
5,FMOD,Fibroblast,colagen_fibral_orga,chr1-203202303-203203152,distal,0.08393939,5.022665,2.547975e-07,10.501871,3.140583e-24,⋯,0.2195122,0.09677419,0.014263074,0.003589375,1.344779,4.774727,0.1227380,1,7,global
6,FMOD,Fibroblast,colagen_fibral_orga,chr1-203350857-203351576,proximal,0.06872044,4.625050,1.872537e-06,5.254400,1.191217e-07,⋯,0.2857143,0.02564103,0.007395668,0.001794688,1.750347,4.177886,0.2600733,1,7,global
7,FMOD,Fibroblast,colagen_fibral_orga,chr1-203368833-203369408,distal,0.07909978,3.613998,1.507555e-04,6.177888,7.506030e-10,⋯,0.2608696,0.00000000,0.009508716,0.002153625,1.598143,3.473896,0.2608696,1,7,global
8,FMOD,Fibroblast,colagen_fibral_orga,chr1-203463638-203463946,distal,0.06127078,3.662711,1.247801e-04,5.089219,2.734281e-07,⋯,0.2500000,0.02857143,0.005810882,0.001435750,1.531553,3.723078,0.2214286,1,7,global
9,FMOD,Fibroblast,colagen_fibral_orga,chr1-203464230-203464459,distal,0.08388339,3.957796,3.782230e-05,4.659539,2.109628e-06,⋯,0.3103448,0.00000000,0.004754358,0.001076813,1.901239,5.252199,0.3103448,1,7,both


In [8]:
write.csv(peak_stats_summary, "objects_AVN_fibroblast/peak_stats_summary_collagen_fibril_organization.csv", row.names = FALSE)

## motif stats: Motif enrichment and footprint scoring
#### Identity Columns
- cluster
- gene
- motif
- TF: Transcription factor that binds motif (if multiple, new row for each)

#### Motif enrichment in linked peaks: motif_enrichment_per_gene() / calculate_enrichments()
logic: 
1. find peaks linked to the gene in the chosen cluster
2. split them into proximal vs distal using regulatorType
3. count how often each motif occurs in those linked peaks
4. compare that to a random background made by replacing each linked peak with a sampled “comparable”(GC content and length) peak from the same peak meta-feature cluster. t-test for enrichment
-> for this: make clusters of comparable peaks, store in object$peaks@meta.features$cluster
##### Proximal motif columns
- proximal.motif_count: The number of occurrences of this motif in proximal peaks linked to the gene in this cluster
- proximal.background_count: The mean motif count in matched random background peak sets for the proximal linked peaks. For each background draw, every linked proximal peak is replaced by a sampled peak from the same object$peaks@meta.features$cluster, then motif counts are computed, and the mean over draws is stored here
- log2FC.proximal: log2(proximal.motif_count / proximal.background_count): log2 enrichment of proximal motif count over matched background. Positive values mean enrichment; zero means no enrichment; negative values mean depletion
- t_stat.proximal: The t-statistic from testing the proximal background distribution against the foreground proximal count using
t.test(col_bg, mu = value_fg, alternative = "less") in effect. Interpreted practically, small p-values support that the background mean is lower than the foreground count, i.e. the motif is enriched in the linked proximal peaks.
- p_value.proximal: The raw p-value for that proximal enrichment test.
- p_adjust.proximal: A simple multiplicity-adjusted proximal p-value, computed as raw p_value.proximal * number_of_motifs, then capped at 1. This is Bonferroni-like, not BH/FDR.

##### Distal motif columns: analog to proximal, but using distal peaks
- distal.motif_count
- distal.background_count
- log2FC.distal = log2(distal.motif_count / distal.background_count)
- t_stat.distal
- p_value.distal
- p_adjust.distal

##### Promoter motif column
- promoter.motif_count: The number of occurrences of this motif in the gene’s promoter sequence, regardless of whether there is an accessible linked peak there. This is computed by get_tf_bindingsites_in_region(), which scans the promoter DNA sequence with the motif PWM. The promoter length: 2000 bp upstream TSS, but it may be extended if the gene has proximal linked peaks farther upstream than 2 kb.

#### Footprint columns: add_motif_stats() -> footprint_stats_test() for each  (gene, cluster, motif)
idea: The foreground footprint is built from motif sites found in peaks linked to the gene; the background footprint distribution is built by repeatedly sampling the same number of motif sites of that motif from elsewhere and rescoring them
core = mean.footprint - mean.flanks
- difference = observed.insertions_normalized - expected.insertions 
- mean.flanks = mean(difference in left + right flank windows)
- mean.footprint = mean(difference in motif-core window)
more negative footprint_score means the motif center is more depleted relative to the flanks, which is the classic “footprint” pattern
values near 0 mean little difference between core and flanks
positive values would mean the core is more accessible than the flanks

observed.insertions_normalized: empirical insertion profile around the motif sites, normalized by its global mean (it is the mean observed Tn5 insertion signal at each relative position, normalized so the average across positions is 1)
expected.insertions: sequence-bias-based expected insertion profile, normalized by flank expectation  (comes from GetExpectedInsertion(), which extracts the DNA sequence around the motif regions, gets the assay’s Tn5 bias vector, and calls FindExpectedInsertions(). That helper computes the expected insertion profile from local sequence composition and Tn5 hexamer bias, then normalizes that expected profile by the mean of the flank positions.)

##### Footprint statistic columns
- footprint_scores: as explained above
- bg_size: how many bg scores where computed
- bg_footprint_mean: The mean of the background footprint scores across those sampled background footprints
- footprint.t_stat: t.test(x = background_scores, mu = foreground_score, alternative = "greater"): So the test is asking whether the background mean score is greater than the foreground score. Because more negative scores are stronger footprints in your setup, a small p-value supports the foreground having a stronger depletion footprint than background
- footprint.p_value: The raw p-value for that footprint-vs-background test.
- footprint.p_value_adj: A per-(gene, cluster) multiplicity-adjusted version of footprint.p_value. In add_motif_stats(), after all motif rows are combined, the code multiplies each raw footprint p-value by the number of motif rows for that same gene and cluster, then caps at 1. So again this is Bonferroni-like, not BH/FDR.

##### Footprint quality / coverage columns
- sd.flanks: The standard deviation of the flank difference values for the foreground footprint. This is a variability measure for the left/right flank regions around the motif
- bg_sd_mean: The mean of the flank standard deviations across the sampled background footprints
- left_flank_nonzero_positions: How many positions in the left flank had nonzero observed insertion counts in the foreground footprint pileup. It is computed from observed.insertions.counts_sum in the footprint plot data
- right_flank_nonzero_positions: same for right flank

In [9]:

motif2TF_df <- data.frame(
  motif = rep(names(object_w_net@misc$motif2TF),
              lengths(object_w_net@misc$motif2TF)),
  TF = unlist(object_w_net@misc$motif2TF, use.names = FALSE),
  stringsAsFactors = FALSE
)

head(motif2TF_df)

,motif,TF
,<chr>,<chr>
1,MA0069.1,PAX6
2,MA0071.1,RORA
3,MA0074.1,RXRA
4,MA0074.1,VDR
5,MA0101.1,REL
6,MA0107.1,RELA


In [10]:

tmp <- object_w_net@misc$motif_stats %>%
  left_join(motif2TF_df, by = "motif")

Warning message in left_join(., motif2TF_df, by = "motif"):
"Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 2 of `x` matches multiple rows in `y`.
ℹ Row 12 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning."


In [11]:
motif_summary_cols <- c(
  "gene",
  "cluster",
  "TF",
  "motif",
  "proximal.motif_count",
  "proximal.background_count",
  "log2FC.proximal",
  "p_adjust.proximal",
  "distal.motif_count",
  "distal.background_count",
  "log2FC.distal",
  "p_adjust.distal",
  "promoter.motif_count",
  "footprint_score",
  "bg_footprint_mean",
  "footprint.p_value_adj",
  "bg_size",
  "sd.flanks",
  "bg_sd_mean",
  "left_flank_nonzero_positions",
  "right_flank_nonzero_positions"
)

In [12]:
motif_summary_col_labels <- c(
  gene = "Gene",
  cluster = "Cluster",
  TF = "TF",
  motif = "Motif",
  proximal.motif_count = "Prox Motif count",
  proximal.background_count = "Prox Bg count",
  log2FC.proximal = "Prox Log2FC",
  p_adjust.proximal = "Prox p-value adj",
  distal.motif_count = "Dist Motif count",
  distal.background_count = "Dist Bg count",
  log2FC.distal = "Dist Log2FC",
  p_adjust.distal = "Dist p-value adj",
  promoter.motif_count = "Prom Motif count",
  footprint_score = "FP Score",
  bg_footprint_mean = "Bg FP Score",
  footprint.p_value_adj = "FP p-value adj",
  bg_size = "Bg Size",
  sd.flanks = "Flank sd",
  bg_sd_mean = "Bg Flank sd",
  left_flank_nonzero_positions = "Left Flank != 0",
  right_flank_nonzero_positions = "Right Flank != 0"
)

In [13]:
colnames(tmp)

[1] "gene"                          "cluster"                      
 [3] "motif"                         "proximal.motif_count"         
 [5] "proximal.background_count"     "log2FC.proximal"              
 [7] "t_stat.proximal"               "p_value.proximal"             
 [9] "p_adjust.proximal"             "distal.motif_count"           
[11] "distal.background_count"       "log2FC.distal"                
[13] "t_stat.distal"                 "p_value.distal"               
[15] "p_adjust.distal"               "promoter.motif_count"         
[17] "footprint_score"               "bg_size"                      
[19] "bg_footprint_mean"             "footprint.t_stat"             
[21] "footprint.p_value"             "footprint.p_value_adj"        
[23] "sd.flanks"                     "bg_sd_mean"                   
[25] "left_flank_nonzero_positions"  "right_flank_nonzero_positions"
[27] "TF"

In [15]:
motif_stats_summary[motif_stats_summary$`Prox Motif count`>0,]

,Gene,Cluster,TF,Motif,Prox Motif count,Prox Bg count,Prox Log2FC,Prox p-value adj,Dist Motif count,Dist Bg count,⋯,Dist p-value adj,Prom Motif count,FP Score,Bg FP Score,FP p-value adj,Bg Size,Flank sd,Bg Flank sd,Left Flank != 0,Right Flank != 0
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
231,AEBP1,Fibroblast,EWSR1-FLI1,MA0149.1,3,1.28,1.2288187,1.139550e-03,2,0.64,⋯,3.500278e-08,0,0.11620,0.089530,1.0e+00,50,1.112,2.135,11,16
232,AEBP1,Fibroblast,INSM1,MA0155.1,1,0.34,1.5563933,2.235673e-09,0,0.12,⋯,1.000000e+00,3,-0.89880,-0.252700,1.0e+00,50,5.297,4.343,3,2
233,AEBP1,Fibroblast,PLAG1,MA0163.1,2,0.62,1.6896599,2.536613e-14,0,0.06,⋯,1.000000e+00,0,0.70210,-0.005837,1.0e+00,50,2.075,3.362,3,4
235,AEBP1,Fibroblast,KLF5,MA0599.1,7,1.96,1.8365013,6.119142e-22,1,0.44,⋯,6.901862e-04,12,-0.34780,-0.401700,1.0e+00,50,1.090,1.384,14,15
236,AEBP1,Fibroblast,ZIC1,MA0696.1,1,0.56,0.8365013,3.491301e-02,1,0.04,⋯,2.343737e-33,0,-0.25170,-0.206900,1.0e+00,50,1.977,2.958,1,6
238,AEBP1,Fibroblast,GLIS2,MA0736.1,1,0.22,2.1844246,1.817694e-13,0,0.04,⋯,1.000000e+00,2,0.16730,-0.543600,1.0e+00,50,2.390,4.117,0,3
239,AEBP1,Fibroblast,KLF16,MA0741.1,5,1.02,2.2933589,5.881183e-26,0,0.24,⋯,1.000000e+00,5,-1.01900,-0.461200,7.2e-05,50,1.657,1.808,11,10
245,AEBP1,Fibroblast,ZNF460,MA1596.1,3,1.36,1.1413558,2.643074e-07,1,0.32,⋯,1.683818e-07,0,-0.39120,-0.212900,1.0e+00,50,1.697,2.347,9,10
247,AEBP1,Fibroblast,ZFP14,MA1972.1,1,0.70,0.5145732,1.000000e+00,1,0.22,⋯,3.563415e-15,4,0.99350,0.063000,1.0e+00,50,1.683,3.127,5,4


In [14]:
motif_stats_summary <- tmp[, motif_summary_cols]
colnames(motif_stats_summary) <- motif_summary_col_labels[motif_summary_cols]

motif_stats_summary

Gene,Cluster,TF,Motif,Prox Motif count,Prox Bg count,Prox Log2FC,Prox p-value adj,Dist Motif count,Dist Bg count,⋯,Dist p-value adj,Prom Motif count,FP Score,Bg FP Score,FP p-value adj,Bg Size,Flank sd,Bg Flank sd,Left Flank != 0,Right Flank != 0
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
ADAMTS2,Fibroblast,RORA,MA0071.1,0,0,NA,1,1,0.06,⋯,4.881223e-29,0,1.244000,0.0810600,1.00000,50,1.645,4.362,7,7
ADAMTS2,Fibroblast,RXRA,MA0074.1,0,0,NA,1,1,0.06,⋯,4.881223e-29,0,0.073580,0.1205000,1.00000,50,2.344,5.007,4,7
ADAMTS2,Fibroblast,VDR,MA0074.1,0,0,NA,1,1,0.06,⋯,4.881223e-29,0,0.073580,0.1205000,1.00000,50,2.344,5.007,4,7
ADAMTS2,Fibroblast,NR1H2,MA0115.1,0,0,NA,1,1,0.12,⋯,1.195292e-21,0,0.004245,-0.4535000,1.00000,50,2.532,4.489,1,3
ADAMTS2,Fibroblast,RXRA,MA0115.1,0,0,NA,1,1,0.12,⋯,1.195292e-21,0,0.004245,-0.4535000,1.00000,50,2.532,4.489,1,3
ADAMTS2,Fibroblast,NFIC,MA0119.1,0,0,NA,1,1,0.22,⋯,4.114275e-10,0,0.785200,-0.7726000,1.00000,50,3.391,4.405,0,1
ADAMTS2,Fibroblast,TLX1,MA0119.1,0,0,NA,1,1,0.22,⋯,4.114275e-10,0,0.785200,-0.7726000,1.00000,50,3.391,4.405,0,1
ADAMTS2,Fibroblast,ZNF354C,MA0130.1,0,0,NA,1,3,1.30,⋯,4.993569e-12,6,0.053350,0.0009304,1.00000,50,1.752,2.845,5,8
ADAMTS2,Fibroblast,EWSR1-FLI1,MA0149.1,0,0,NA,1,5,1.52,⋯,1.729579e-11,0,0.130200,0.1738000,1.00000,50,1.283,2.041,14,13


In [19]:
write.csv(motif_stats_summary, "objects_AVN_fibroblast/motif_stats_summary_collagen_fibril_organization.csv", row.names = FALSE)

## network

In [20]:
str(object_w_net@misc$context_subNetwork, max.level = 1)

List of 5
 $ colagen_fibral_orga                     :'data.frame':	935 obs. of  6 variables:
 $ colagen_fibral_orga_omni_path           :'data.frame':	927 obs. of  6 variables:
 $ colagen_fibral_orga_omni_path_cl_unsp   :'data.frame':	927 obs. of  6 variables:
 $ colagen_fibral_orga_omni_path_full_prior:'data.frame':	1138 obs. of  6 variables:
 $ colagen_fibral_orga_complete            :'data.frame':	2093 obs. of  6 variables:


In [21]:
head(object_w_net@misc$context_subNetwork$colagen_fibral_orga_complete)

,from,to,color,priorTF,reg_type,in.prom
,<chr>,<chr>,<chr>,<lgl>,<int>,<int>
ADAMTS7.1,HR,ADAMTS7,black,TRUE,2,1
ADAMTS7.2,HTATIP2,ADAMTS7,black,TRUE,2,1
ADAMTS7.3,SSRP1,ADAMTS7,black,TRUE,2,1
ADAMTS7.4,TFAP2A,ADAMTS7,black,TRUE,2,1
ADAMTS7.5,ZBTB16,ADAMTS7,black,TRUE,2,1
AEBP1.1,SP1,AEBP1,red,TRUE,1,3


In [22]:
unique(object_w_net@misc$context_subNetwork$colagen_fibral_orga_complete$to)

[1] "ADAMTS7"  "AEBP1"    "ANXA2"    "ATP7A"    "BMP1"     "COL11A1" 
 [7] "COL11A2"  "COL1A1"   "COL1A2"   "COL2A1"   "COL3A1"   "COL4A1"  
[13] "COL4A2"   "COL4A3"   "COL4A4"   "COL5A1"   "COL5A2"   "COL5A3"  
[19] "COL6A1"   "COL7A1"   "COMP"     "CYP1B1"   "DDR2"     "DPT"     
[25] "EFEMP2"   "ERO1A"    "EXT1"     "FMOD"     "FOXC1"    "FOXC2"   
[31] "GREM1"    "LOX"      "LOXL2"    "LOXL4"    "LUM"      "MMP11"   
[37] "NF1"      "P3H4"     "P4HA1"    "PLOD1"    "PLOD2"    "PXDN"    
[43] "RB1"      "SCX"      "SERPINF2" "SERPINH1" "SFRP2"    "TGFB2"   
[49] "TGFBR1"   "TLL1"     "TNXB"     "VPS33B"   "COL12A1"  "COL14A1" 
[55] "COLGALT2"

In [51]:
green_counts_by_to <- object_w_net@misc$context_subNetwork$colagen_fibral_orga_complete %>%
    group_by(to) %>%
    summarise(
        priorTF_count = sum(priorTF == TRUE, na.rm = TRUE),
        green_count = sum(color == "green", na.rm = TRUE),
        yellow_count = sum(color == "yellow", na.rm = TRUE),
        red_count = sum(color == "red", na.rm = TRUE),
        grey_count = sum(color == "grey", na.rm = TRUE),
        black_count = sum(color == "black", na.rm = TRUE),
        .groups = "drop"
    ) %>%
    arrange(desc(green_count))

top_green_genes <- green_counts_by_to[1:9,]$to

priorTF_count <- object_w_net@misc$context_subNetwork$colagen_fibral_orga_complete %>%
    group_by(to) %>%
    summarise(
        priorTF_count = sum(priorTF == TRUE, na.rm = TRUE),
        green_count = sum(color == "green", na.rm = TRUE),
        yellow_count = sum(color == "yellow", na.rm = TRUE),
        red_count = sum(color == "red", na.rm = TRUE),
        grey_count = sum(color == "grey", na.rm = TRUE),
        black_count = sum(color == "black", na.rm = TRUE),
        .groups = "drop"
    ) %>%
    arrange(desc(priorTF_count))

top_priorTF_genes <- priorTF_count[1:9,]$to
chat_imp <- c("COL1A1", "COL1A2", "COL5A1", "COL5A2", "BMP1", "ADAMTS2", "LOX", "COL3A1", "FMOD", "LUM", "TNXB", "COMP", "PLOD1", "SERPINH1", "COL11A1")

res <- green_counts_by_to[green_counts_by_to$to %in% union(union(top_green_genes, top_priorTF_genes), chat_imp),]


In [52]:
subset_genes <- res$to
sub_net <- object_w_net@misc$context_subNetwork$colagen_fibral_orga_complete %>%
    filter(to %in% subset_genes)

head(sub_net)

,from,to,color,priorTF,reg_type,in.prom
,<chr>,<chr>,<chr>,<lgl>,<int>,<int>
AEBP1.1,SP1,AEBP1,red,TRUE,1,3
AEBP1.2,SP3,AEBP1,green,TRUE,1,3
AEBP1.3,INSM1,AEBP1,red,FALSE,1,3
AEBP1.4,ZIC1,AEBP1,red,FALSE,1,1
AEBP1.5,KLF16,AEBP1,green,FALSE,1,3
AEBP1.6,ZNF143,AEBP1,green,FALSE,2,1


In [54]:

# assumed columns: from, to, color, priorTF, reg_type, in.prom
df <- sub_net
links <- df %>%
  transmute(
    source = from,
    target = to,
    edge_color = color,
    edge_width = reg_type,
    edge_dash = ifelse(in.prom == 1, "4,4", "0")
  )

# node table
from_nodes <- df %>%
  distinct(name = from, priorTF) %>%
  mutate(type = "from")

to_nodes <- df %>%
  distinct(name = to) %>%
  mutate(priorTF = NA, type = "to")

nodes <- bind_rows(from_nodes, to_nodes) %>%
  group_by(name) %>%
  summarise(
    priorTF = dplyr::first(na.omit(priorTF)),
    type = ifelse(any(type == "from"), "from", "to"),
    .groups = "drop"
  ) %>%
  mutate(
    node_fill = case_when(
      type == "to" ~ "#cccccc",          # default for target nodes
      priorTF %in% TRUE ~ "#ff7f0e",     # prior TF
      priorTF %in% FALSE ~ "#1f77b4",    # not prior TF
      TRUE ~ "#999999"
    )
  )

# convert source/target names to node indices for D3
nodes <- nodes %>% mutate(id = row_number() - 1)

links <- links %>%
  left_join(nodes %>% select(source = name, source_id = id), by = "source") %>%
  left_join(nodes %>% select(target = name, target_id = id), by = "target") %>%
  transmute(
    source = source_id,
    target = target_id,
    edge_color,
    edge_width,
    edge_dash
  )

graph <- list(
  nodes = nodes,
  links = links
)


In [61]:
print(length(sub_net[sub_net$color == "green",]$to))
print(length(sub_net[sub_net$color == "yellow",]$to))
print(length(sub_net[sub_net$color == "red",]$to))
print(length(sub_net[sub_net$color == "grey",]$to))
print(length(sub_net[sub_net$color == "black",]$to))

[1] 206
[1] 402
[1] 490
[1] 0
[1] 267


In [55]:
graph

name,priorTF,type,node_fill,id
<chr>,<lgl>,<chr>,<chr>,<dbl>
AEBP1,NA,to,#cccccc,0
AHR,TRUE,from,#ff7f0e,1
AIRE,TRUE,from,#ff7f0e,2
AR,TRUE,from,#ff7f0e,3
ARGFX,FALSE,from,#1f77b4,4
ARID4A,TRUE,from,#ff7f0e,5
ARID5A,TRUE,from,#ff7f0e,6
ARNT,FALSE,from,#1f77b4,7
ARNT2,FALSE,from,#1f77b4,8


In [63]:
library(dplyr)
library(jsonlite)


Attaching package: 'jsonlite'


The following object is masked from 'package:purrr':

    flatten




In [64]:

write_json(graph, "objects_AVN_fibroblast/colagen_fibral_orga_subnet.json", auto_unbox = TRUE, pretty = TRUE)